# Challenge 5: The eval-driven loop

Evals earn their keep when they drive changes. The loop:

1. Run evals on the current agent. (Challenge 4 was this.)
2. Read the failures and form a hypothesis.
3. Change **one thing**.
4. Rerun the **same** evals and compare experiments.

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import Client, uuid7

import lab_eval
from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS, build_agent, extract_tool_calls

client = Client()
DATASET = lab_eval.dataset_name()
print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", DATASET)

from eval_lab.evaluators import exact_match, extra_steps, in_order, make_run_agent, missing_steps

The evaluators and run function are the ones you wrote in challenge 4, imported so the baseline and the fix are scored by identical code.

## Error analysis

Scores say how often the agent fails, not why. Error analysis is the step in between: look at every failing row, name the cause, and count the causes. The biggest count tells you what to fix first.

This cell lists each failing row of your `trajectory` experiment with its expected and actual tool calls, labels each one (`skipped ...`, `extra ...` or `wrong order`), and counts them.

In [ ]:
lab_eval.failure_modes("trajectory")

Read the counts. One cause usually dominates, and that is the hypothesis. If you see a different pattern from the one described below, trust your table: the model's behavior varies from run to run.

## The hypothesis

If your top cause is `skipped check_refund_eligibility`: the prompt never required the check. The fix is a prompt change: require the check before any refund, and refund only eligible orders.

The next cell holds this code:

```python
SYSTEM_PROMPT_V2 = (
    "You are an order-support assistant. Follow this procedure exactly.\n"
    "1. For any question about a specific order, call lookup_order first.\n"
    "2. Before issuing any refund, you MUST call check_refund_eligibility for that order.\n"
    "3. Call issue_refund only if the order is eligible. If it is not eligible, "
    "explain why and do not issue a refund.\n"
    "4. For policy questions (returns, shipping, warranty), call search_faq and answer from its result."
)
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
SYSTEM_PROMPT_V2 = (
    "You are an order-support assistant. Follow this procedure exactly.\n"
    "1. For any question about a specific order, call lookup_order first.\n"
    "2. Before issuing any refund, you MUST call check_refund_eligibility for that order.\n"
    "3. Call issue_refund only if the order is eligible. If it is not eligible, "
    "explain why and do not issue a refund.\n"
    "4. For policy questions (returns, shipping, warranty), call search_faq and answer from its result."
)

## Rerun the same evals

Only the system prompt changes. Same dataset, same evaluators, new experiment prefix `fixed`.

In [ ]:
agent_v2 = build_agent(SYSTEM_PROMPT_V2)

results = client.evaluate(
    make_run_agent(agent_v2),
    data=DATASET,
    evaluators=[exact_match, extra_steps, missing_steps, in_order],
    experiment_prefix="fixed",
    max_concurrency=2,
)
print("experiment:", results.experiment_name)

## Compare experiments

One row per experiment on this dataset. Read down the `exact_match` column: baseline `trajectory-...` versus `fixed-...`.

In [ ]:
lab_eval.compare(prefixes=("first-eval", "single-step", "llm-judge", "trajectory", "fixed"),
                 keys=["keyword_match", "tool_selection", "correctness",
                       "exact_match", "extra_steps", "missing_steps", "in_order"])

Run error analysis again, on the fixed experiment, to see what is left.

In [ ]:
lab_eval.failure_modes("fixed")

**Result:** `fixed` has a higher `exact_match` than `trajectory`, and `missing_steps` at or near 0. For a side-by-side view, open your dataset in LangSmith, select both experiments, and click **Compare**.

Two cautions. LLM output varies, so rerun if one row flips. And a higher average on 8 examples is a signal, not a proof: a real dataset grows every time production finds a new failure.